# AMZ FBA Inventory Ledger -> `punlabs.AMZSales.PL-AMZSales-INVLedger`

Daily load of `GET_LEDGER_SUMMARY_VIEW_DATA` from SP-API.

**What was wrong with the previous version** (see `docs/RUNBOOK.md` §6):

1. It crashed with `EmptyDataError: No columns to parse from file` whenever
   Amazon returned a report with no rows. An empty report is a **zero-byte
   document**, so `pd.read_csv` raised before the `if df.empty:` guard could
   run - that branch was unreachable.
2. It still imported `google.cloud.secretmanager`, the module that caused the
   2026-08-25 outage. On a scheduled Colab Enterprise run this raises
   `ModuleNotFoundError` in the first cell, whatever else is fixed.
3. `WRITE_APPEND` with no key meant every manual re-run duplicated a day.
4. The poll loop had no timeout and re-minted an LWA token every 30 seconds.

All four are fixed below.

In [ ]:
# --- Cell 1: bootstrap -------------------------------------------------------
# Colab Enterprise runtimes start empty, so pull the pipeline library in. Pin to
# a tag or commit once this is scheduled, so a runtime restart cannot silently
# pick up an unreviewed main.
import subprocess, sys, pathlib

REPO_URL = "https://github.com/RGDub/RGDub.git"
REPO_REF = "main"
REPO_DIR = pathlib.Path("/tmp/rgdub")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("pipeline library ready")

In [ ]:
# --- Cell 2: preflight -------------------------------------------------------
# Fail here, loudly, if credentials are unreachable. A scheduled run that cannot
# authenticate must abort before it can write a partial or empty load.
#
# NOTE the import: `pipelines.lib.secrets`, NOT `google.cloud.secretmanager`.
# The latter is no longer on the Colab Enterprise base image and is what took
# the pipeline down on 2026-08-25. See docs/RUNBOOK.md §1.
from pipelines.lib.secrets import preflight
from pipelines.lib.heartbeat import run_logged
from pipelines.lib.spapi_reports import LwaTokenProvider, fetch_report

SECRET_IDS = ["amz-lwa-client-id", "amz-lwa-client-secret", "amz-refresh-token"]
preflight(SECRET_IDS)
print(f"preflight OK - {len(SECRET_IDS)} secrets readable")

In [ ]:
# --- Cell 3: configuration ---------------------------------------------------
import datetime
import pandas as pd
from google.cloud import bigquery

PROJECT_ID     = "punlabs"
DATASET_ID     = "AMZSales"
TABLE_ID       = "PL-AMZSales-INVLedger"
MARKETPLACE_ID = "ATVPDKIKX0DER"
TABLE_REF      = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

REPORT_TYPE = "GET_LEDGER_SUMMARY_VIEW_DATA"
REPORT_OPTIONS = {
    "aggregateByLocation": "COUNTRY",
    "aggregatedByTimePeriod": "DAILY",
}

# Ledger data settles 24-72h after the fact, so the normal window ends 2 days
# back. The window is a *range*, not a single day: a range costs the same, and
# because the load below is keyed by date it simply re-states any day that has
# since been corrected. That makes routine runs self-healing.
NORMAL_LOOKBACK_DAYS = 3

# If the normal window comes back empty, retry once over a wider one before
# concluding there is no data. This distinguishes "the ledger has not settled"
# from "this seller genuinely had no inventory movement", which the old code
# could not tell apart - it just crashed either way.
WIDE_LOOKBACK_DAYS = 14


def utc_window(days_back_start, days_back_end):
    """Whole-day UTC window, end-exclusive at midnight.

    The previous version asked for 00:00:00 .. 23:59:59 on one day. The summary
    view snaps to aggregation-period boundaries, so a window that does not cover
    a whole period can match nothing. Midnight-to-midnight avoids that.
    """
    today = datetime.datetime.now(datetime.timezone.utc).date()
    start = today - datetime.timedelta(days=days_back_start)
    end   = today - datetime.timedelta(days=days_back_end)
    fmt = "%Y-%m-%dT00:00:00Z"
    return start.strftime(fmt), end.strftime(fmt)


print(f"Target: {TABLE_REF}")

In [ ]:
# --- Cell 4: extract ---------------------------------------------------------
tokens = LwaTokenProvider()

start_time, end_time = utc_window(NORMAL_LOOKBACK_DAYS, 1)
result = fetch_report(
    tokens,
    report_type=REPORT_TYPE,
    marketplace_ids=[MARKETPLACE_ID],
    data_start_time=start_time,
    data_end_time=end_time,
    report_options=REPORT_OPTIONS,
)
print(result.describe())

if result.is_empty:
    print(f"\nRetrying over a {WIDE_LOOKBACK_DAYS}-day window to tell a settlement "
          "lag apart from a genuinely empty ledger...")
    start_time, end_time = utc_window(WIDE_LOOKBACK_DAYS, 1)
    result = fetch_report(
        tokens,
        report_type=REPORT_TYPE,
        marketplace_ids=[MARKETPLACE_ID],
        data_start_time=start_time,
        data_end_time=end_time,
        report_options=REPORT_OPTIONS,
    )
    print(result.describe())

df = result.frame

In [ ]:
# --- Cell 5: transform -------------------------------------------------------
if df.empty:
    # Not an error. Report it and skip the load rather than writing nothing and
    # calling it success - the heartbeat in cell 7 records rows_written=0.
    print("No ledger rows for either window; nothing to load.")
else:
    df.columns = df.columns.str.strip()

    COLUMN_MAPPING = {
        "Customer Shipments":        "Shipments",
        "Customer Returns":          "CustomerReturns",
        "Vendor Returns":            "VendorReturns",
        "Warehouse Transfer In/Out": "WhseTransfers",
        "Other Events":              "Adjustments",
    }
    # Surface a mapping that no longer matches the report rather than letting it
    # no-op: Amazon renames report columns from time to time, and a silent
    # no-op here would load NULLs into the mapped columns.
    missing_sources = [c for c in COLUMN_MAPPING if c not in df.columns]
    if missing_sources:
        print(f"WARNING: report has no column(s) {missing_sources} - the rename "
              "map is stale. Check the report's current column set before trusting "
              "this load.")
    df = df.rename(columns=COLUMN_MAPPING)

    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"]).dt.date
    if "Store" in df.columns:
        df["Store"] = df["Store"].astype(str)

    def extract_parent_sku(sku):
        if pd.isna(sku):
            return sku
        clean = str(sku).replace("amzn.gr.", "")
        parts = clean.split("-")
        return f"{parts[0]}-{parts[1]}" if len(parts) > 2 else clean

    df["Parent SKU"] = df["MSKU"].apply(extract_parent_sku) if "MSKU" in df.columns else "UNKNOWN"

    print(f"{len(df)} rows ready, {df['Date'].nunique()} distinct dates: "
          f"{df['Date'].min()} .. {df['Date'].max()}")
    display(df.head())

In [ ]:
# --- Cell 6: schema check ----------------------------------------------------
# The load previously failed opaquely when the report's shape drifted from the
# table's. Compare up front and print the difference.
if not df.empty:
    client = bigquery.Client(project=PROJECT_ID)
    table = client.get_table(TABLE_REF)
    table_cols = {f.name for f in table.schema}
    frame_cols = set(df.columns)

    only_in_frame = sorted(frame_cols - table_cols)
    only_in_table = sorted(table_cols - frame_cols)

    if only_in_frame:
        raise ValueError(
            f"Report columns absent from {TABLE_ID}: {only_in_frame}. "
            "Add them to the table, or extend COLUMN_MAPPING in cell 5. Loading "
            "as-is would fail mid-job and leave the day partially written."
        )
    if only_in_table:
        print(f"NOTE: table columns not present in this report (will be NULL): {only_in_table}")
    print(f"Schema check OK - {len(frame_cols)} columns align.")

In [ ]:
# --- Cell 7: idempotent load -------------------------------------------------
# The previous version used a bare WRITE_APPEND, so every manual re-run - and
# there have been many since the August outage - appended a second copy of the
# same day. Delete the dates we are about to write, then append them. Safe to
# run as often as you like.
with run_logged("amz_fba_inv_ledger") as run:
    if df.empty:
        print("Nothing to load; recording a zero-row run.")
    else:
        client = bigquery.Client(project=PROJECT_ID)
        load_dates = sorted(df["Date"].unique())

        delete_sql = f"""
            DELETE FROM `{TABLE_REF}`
            WHERE Date IN UNNEST(@dates)
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ArrayQueryParameter("dates", "DATE", load_dates)]
        )
        delete_job = client.query(delete_sql, job_config=job_config)
        delete_job.result()
        print(f"Cleared {delete_job.num_dml_affected_rows} existing rows for "
              f"{len(load_dates)} date(s).")

        load_job = client.load_table_from_dataframe(
            df, TABLE_REF,
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"),
        )
        load_job.result()
        run.rows_written = len(df)
        print(f"SUCCESS - loaded {len(df)} rows into {TABLE_ID}.")